In [14]:
import pickle
import pandas as pd
import numpy as np
from src.utils import load_env, set_seed, get_logger, load_json
from case_study.utils import set_matplot_style
from src.data_loading import DatasetLoader
from src.experiment_config import ExperimentConfig
from experiments.cluster_lr import load_filtered_raw_data

DATASET = "podcasts"
LABELED_ONLY = False

CLUSTER_TEC = "pckmeans" # or pckmeans
K_VAL = 225

# only applies if pckmeans set 
PCK_WEIGHT = 0.1

EMB_NAME = "mets_only_qwen_fe"
ABLATION_NAME = None

env_vars = load_env()
logger = get_logger("cluster_lr_eval")
set_seed(env_vars["RANDOM_SEED"])

In [15]:
# get data loader 
config = ExperimentConfig(
        task_type="cluster_lr",
        task_name="",
        dataset=DATASET,
        labeled_only=LABELED_ONLY,
        logger=logger,
        env_vars=env_vars
    )
data_loader = DatasetLoader(config)

In [16]:
# load dataset target group options / definitions
tg_def_path = f"{env_vars['RESULTS_DIR']}/clean_target_groups/llm/{config.dataset_out_name}_target_group_cleanup.json"
tg_data = load_json(tg_def_path)['data']

In [17]:
# load metaphor embedding data
# met_emb_data_path = f"{env_vars["RESULTS_DIR"]}/gen_embeddings/sbert/{DATASET}_{EMB_NAME}_features.pkl"
met_emb_data_path = f"{env_vars["RESULTS_DIR"]}/gen_embeddings/sbert/{config.dataset_out_name}_{EMB_NAME}_embeddings_features.pkl"

with open(met_emb_data_path, "rb") as f:
    met_emb_data = pickle.load(f)

target_groups = {"person": {}, "place": {}, "thing": {}, "organization": {}}
for k, v in met_emb_data.items():
    if 'features' in v:
        ng = v['features']['noun_classification']
        tg = v['features']['target_group']
        tg_def = ""
        if tg not in ['Audience', 'Conversation Partner', 'Other']:
            tg_def = tg_data[ng]['annotation']['clean_groups'][tg]
        target_groups[ng][tg] = tg_def

with open(f"{config.dataset_out_name}_tg.txt", 'w+') as f:
    f.write(str(target_groups))
